##CREATE GOLD VIEWS

In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS olist_lakehouse.gold
""")

###dim_customers

In [0]:
spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_customers AS

WITH customer_data AS (

    SELECT 

        c.customer_unique_id,

        COALESCE(
            g.geolocation_city,
            c.customer_city
        ) AS city,

        c.customer_state AS state,

        c.customer_zip_code_prefix AS zip_code,

        g.geolocation_lat AS latitude,
        g.geolocation_lng AS longitude,

        ROW_NUMBER() OVER (
            PARTITION BY c.customer_unique_id
            ORDER BY c.customer_zip_code_prefix
        ) AS row_num

    FROM olist_lakehouse.silver.olist_customers_dataset c

    LEFT JOIN
        olist_lakehouse.silver.olist_geolocation_dataset g
    ON
        c.customer_zip_code_prefix =
        g.geolocation_zip_code_prefix

)

SELECT

    ROW_NUMBER() OVER (
        ORDER BY customer_unique_id
    ) AS customer_key,

    customer_unique_id AS customer_id,

    city,
    state,
    zip_code,

    latitude,
    longitude

FROM customer_data

WHERE row_num = 1

""")

###dim_sellers

In [0]:
spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_sellers AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY s.seller_id
    ) AS seller_key,

    s.seller_id,

    COALESCE(
        g.geolocation_city,
        s.seller_city
    ) AS city,

    s.seller_state AS state,

    s.seller_zip_code_prefix AS zip_code,

    g.geolocation_lat AS latitude,
    g.geolocation_lng AS longitude

FROM olist_lakehouse.silver.olist_sellers_dataset s

LEFT JOIN
    olist_lakehouse.silver.olist_geolocation_dataset g
ON
    s.seller_zip_code_prefix =
    g.geolocation_zip_code_prefix

""")

###dim_products

In [0]:
spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_products AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY p.product_id
    ) AS product_key,

    p.product_id,

    CASE

        WHEN t.product_category_name IS NULL
             AND p.product_category_name = 'n/a'

        THEN p.product_category_name

        WHEN t.product_category_name IS NULL

        THEN CONCAT(
            p.product_category_name,
            ' (Translate)'
        )

        ELSE t.product_category_name_english

    END AS product_category,

    p.product_weight_g,
    p.product_length_cm,
    p.product_height_cm,
    p.product_width_cm,

    p.product_name_lenght,
    p.product_description_lenght,
    p.product_photos_qty

FROM olist_lakehouse.silver.olist_products_dataset p

LEFT JOIN
    olist_lakehouse.silver.product_category_name_translation t
ON
    p.product_category_name =
    t.product_category_name

""")

###dim_payments

In [0]:
spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_payments AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY order_id
    ) AS payment_key,

    order_id,

    SUM(
        CASE WHEN payment_type = 'credit_card'
        THEN 1 ELSE 0 END
    ) AS credit_card_payment_count,

    SUM(
        CASE WHEN payment_type = 'credit_card'
        THEN payment_value ELSE 0 END
    ) AS credit_card_payment,

    SUM(
        CASE WHEN payment_type = 'boleto'
        THEN payment_value ELSE 0 END
    ) AS boleto_payment,

    SUM(
        CASE WHEN payment_type = 'voucher'
        THEN payment_value ELSE 0 END
    ) AS voucher_payment,

    SUM(
        CASE WHEN payment_type = 'debit_card'
        THEN payment_value ELSE 0 END
    ) AS debit_card_payment,

    SUM(payment_value)
        AS total_payment

FROM olist_lakehouse.silver.olist_order_payments_dataset

GROUP BY order_id

""")

###dim_reviews

In [0]:
spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.dim_reviews AS

SELECT

    ROW_NUMBER() OVER (
        ORDER BY order_id
    ) AS review_key,

    order_id,

    AVG(review_score)
        AS average_review_score,

    CONCAT_WS(
        ', ',
        COLLECT_LIST(review_comment_title)
    ) AS review_titles,

    CONCAT_WS(
        ', ',
        COLLECT_LIST(review_comment_message)
    ) AS review_messages

FROM olist_lakehouse.silver.olist_order_reviews_dataset

GROUP BY order_id

""")

###fact_orders

In [0]:
spark.sql("""

CREATE OR REPLACE VIEW olist_lakehouse.gold.fact_orders AS

WITH cte_orders_dataset AS (

    SELECT

        o.order_id,

        c.customer_unique_id,

        o.order_status,

        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        o.dq_invalid_approval_timestamp_flag,
        o.dq_invalid_carrier_timestamp_flag,
        o.dq_invalid_customer_delivery_timestamp_flag,
        o.dq_invalid_estimated_delivery_timestamp_flag

    FROM olist_lakehouse.silver.olist_orders_dataset o

    LEFT JOIN olist_lakehouse.silver.olist_customers_dataset c
        ON o.customer_id = c.customer_id

),

cte_orders AS (

    SELECT

        i.order_id,

        o.customer_unique_id
            AS customer_id,

        i.product_id,
        i.seller_id,

        i.price,

        i.freight_value
            AS shipping_cost,

        o.order_status,

        i.shipping_limit_date,

        o.order_purchase_timestamp,
        o.order_approved_at,
        o.order_delivered_carrier_date,
        o.order_delivered_customer_date,
        o.order_estimated_delivery_date,

        o.dq_invalid_approval_timestamp_flag,
        o.dq_invalid_carrier_timestamp_flag,
        o.dq_invalid_customer_delivery_timestamp_flag,
        o.dq_invalid_estimated_delivery_timestamp_flag

    FROM olist_lakehouse.silver.olist_order_items_dataset i

    LEFT JOIN cte_orders_dataset o
        ON i.order_id = o.order_id

)
,cte_deduplicate_orders AS(
SELECT
*
FROM
(SELECT
    order_id,
    customer_id,
    product_id,
    seller_id,
    price,
    shipping_cost,
    order_status,
    shipping_limit_date,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    dq_invalid_approval_timestamp_flag,
    dq_invalid_carrier_timestamp_flag,
    dq_invalid_customer_delivery_timestamp_flag,
    dq_invalid_estimated_delivery_timestamp_flag,
    ROW_NUMBER() OVER (PARTITION BY order_id,
    customer_id,
    product_id,
    seller_id,
    price,
    shipping_cost,
    order_status,
    shipping_limit_date,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    dq_invalid_approval_timestamp_flag,
    dq_invalid_carrier_timestamp_flag,
    dq_invalid_customer_delivery_timestamp_flag,
    dq_invalid_estimated_delivery_timestamp_flag ORDER BY order_id  ) flag1,
    COUNT(*) OVER (PARTITION BY order_id,
    customer_id,
    product_id,
    seller_id,
    price,
    shipping_cost,
    order_status,
    shipping_limit_date,
    order_purchase_timestamp,
    order_approved_at,
    order_delivered_carrier_date,
    order_delivered_customer_date,
    order_estimated_delivery_date,
    dq_invalid_approval_timestamp_flag,
    dq_invalid_carrier_timestamp_flag,
    dq_invalid_customer_delivery_timestamp_flag,
    dq_invalid_estimated_delivery_timestamp_flag ORDER BY order_id  ) flag2
FROM cte_orders)t WHERE flag1=1

)

SELECT
    o.order_id,
    c.customer_key,
    s.seller_key,
    p.product_key,
    r.review_key,
    pa.payment_key,
    o.price,
    o.shipping_cost,
    o.order_status,
    o.shipping_limit_date,
    o.order_purchase_timestamp,
    o.order_approved_at,
    o.order_delivered_carrier_date,
    o.order_delivered_customer_date,
    o.order_estimated_delivery_date,
    o.dq_invalid_approval_timestamp_flag,
    o.dq_invalid_carrier_timestamp_flag,
    o.dq_invalid_customer_delivery_timestamp_flag,
    o.dq_invalid_estimated_delivery_timestamp_flag
FROM cte_deduplicate_orders o

LEFT JOIN olist_lakehouse.gold.dim_customers c
    ON o.customer_id = c.customer_id

LEFT JOIN olist_lakehouse.gold.dim_sellers s
    ON o.seller_id = s.seller_id

LEFT JOIN olist_lakehouse.gold.dim_products p
    ON o.product_id = p.product_id

LEFT JOIN olist_lakehouse.gold.dim_reviews r
    ON o.order_id = r.order_id
LEFT JOIN olist_lakehouse.gold.dim_payments pa
    ON o.order_id = pa.order_id;


""")

###validation

In [0]:
display(
    spark.sql("""
    SELECT * 
    FROM olist_lakehouse.gold.fact_orders
    LIMIT 10
    """)
)